In [ ]:
#| hide
#| eval: false
! [ -e /content ] && pip install -Uqq xcube  # upgrade xcube on colab

In [ ]:
#| export
# from fastai.torch_imports import *
# from fastai.torch_core import *
# from fastai.callback.core import *
# from fastcore.all import *
from xcube.imports import *
from datasets import load_dataset, load_from_disk 
from nltk.tokenize import sent_tokenize, word_tokenize

In [ ]:
#| hide
from nbdev.showdoc import *
%load_ext autoreload
%autoreload 2

In [ ]:
#| default_exp text.topicmodel

# Topic Modeling
> with the excellent library BERTopic

In this notebook we perform topic modeling with the python library [BERTopic](https://maartengr.github.io/BERTopic/index.html)

In [ ]:
#| hide
from huggingface_hub import notebook_login, logout
notebook_login()

In [ ]:
#| export
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, OpenAI, PartOfSpeech

from bertopic.vectorizers import ClassTfidfTransformer
import openai

In [ ]:
#| export
def get_topic_model(embedding_model="all-MiniLM-L6-v2",
                    umap_n_neighbors=15, umap_n_components=5, umap_min_dist=0.0, umap_metric='cosine',
                    hdbscan_min_cluster_size=15, hdbscan_metric='euclidean', hdbscan_cluster_selection_method='eom',
                    vectorizer_stop_words="english",
                    representation_model=None,
                    random_state=None,
                    low_memory=False,
                    openai_API_KEY=None,
                    **kwargs):
    
    # Step 1 - Extract embeddings
    embedding_model = SentenceTransformer(embedding_model)
    
    # Step 2 - Reduce dimensionality
    umap_model = UMAP(n_neighbors=umap_n_neighbors, n_components=umap_n_components, min_dist=umap_min_dist, metric=umap_metric, random_state=random_state, low_memory=low_memory)

    # Step 3 - Cluster reduced embeddings
    hdbscan_model = HDBSCAN(min_cluster_size=hdbscan_min_cluster_size, metric=hdbscan_metric, cluster_selection_method=hdbscan_cluster_selection_method, prediction_data=True)

    # Step 4 - Tokenize topics
    vectorizer_model = CountVectorizer(stop_words=vectorizer_stop_words, min_df=2, ngram_range=(1,2))

    # Step 5 - Create topic representation
    ctfidf_model = ClassTfidfTransformer()

    # Step 6 - (Optional) Fine-tune topic representations with a `bertopic.representation` model
    # KeyBERT
    keybert_model = KeyBERTInspired()

    # GPT-3.5
    if openai_API_KEY:
        client = openai.OpenAI(api_key=openai_API_KEY)
        prompt = """
        I have a topic that contains the following documents: 
        [DOCUMENTS]
        The topic is described by the following keywords: [KEYWORDS]

        Based on the information above, extract a short but highly descriptive topic label of at most 5 words. Make sure it is in the following format:
        topic: <topic label>
        """
        openai_model = OpenAI(client, model="gpt-4o-mini", exponential_backoff=True, chat=True, prompt=prompt)
    representation_model = {
        "KeyBERT": keybert_model,
        # "OpenAI": openai_model if openai_API_KEY else None,  # Uncomment if you will use OpenAI
    }

    # Conditionally add OpenAI model if the API key is available
    if openai_API_KEY:
        representation_model["OpenAI"] = openai_model
    
    # All steps together
    topic_model = BERTopic(embedding_model=embedding_model,
                           umap_model=umap_model,
                           hdbscan_model=hdbscan_model,
                           vectorizer_model=vectorizer_model,
                           ctfidf_model=ctfidf_model,
                           representation_model=representation_model,
                           verbose=True,
                           low_memory=low_memory,
                           **kwargs
                           )

    return embedding_model, topic_model
 

## Data

Get the data to perform topic modeling on:

In [ ]:
trn_dset = load_dataset("deb101/lf_amazon_131k", split='train')
# trn_dset = load_from_disk(f"{Path.cwd().parent}/scripts/temp/lf_amazon_131k") # to load locally
trn_dset

In [ ]:
splt_sz = 2000
titles = trn_dset['title'][:splt_sz]
contents = trn_dset['content'][:splt_sz]
docs = [f'{t} \n {c}' for t,c in  zip(titles, contents)]
# docs = [sent_tokenize(doc) for doc in docs]
# docs = [sentence for doc in docs for sentence in doc]
len(docs)

## Training

In [ ]:
embedding_model, topic_model = get_topic_model(embedding_model="all-MiniLM-L6-v2", 
                                                       openai_API_KEY=None, 
                                                       language='multilingual', 
                                                       calculate_probabilities=False,
                                                       low_memory=False,
                                                       seed_topic_list=None)

In [ ]:
%%time
# Pre-calculate embeddings
embeddings = embedding_model.encode(docs, show_progress_bar=True)

In [ ]:
topics, probs = topic_model.fit_transform(docs, embeddings) # topics and the corresponding probabilities

In [ ]:
topic_model.topic_labels_

In [ ]:
probs.shape

In [ ]:
# Create topic model
topic_model = BERTopic(calculate_probabilities=True).fit(docs)
topics, probs = topic_model.transform(docs)

In [ ]:
topic_model.get_topics().keys()

In [ ]:
len(topics), topic_model.probabilities_[999], topics[999]

In [ ]:
topic_model.probabilities_[789], topics[789]

In [ ]:
outlier_count = df_topics[df_topics['Topic']==-1].Count.values[0]
outlier_count

In [ ]:
df_topics = topic_model.get_topic_info()
df_topics.head()

In [ ]:
#| export
def get_topicsmap(df_topics):
    """
    Creates a dictionary mapping topics to their corresponding names.

    Args:
    - df_topics (pandas.DataFrame): A DataFrame containing topic information.
        It should have at least two columns: 'Topic' and 'Name'.

    Returns:
    - dict: A dictionary where the keys are topic numbers and the values are
        the corresponding names of the topics.
    """
    return dict(zip(df_topics['Topic'], df_topics['CustomName']))


In [ ]:
# Example usage
topics_map = get_topicsmap(df_topics)
topics_map

In [ ]:
#| export
def print_representative_topics(df, topic_num):
    """
    Print representative topics for a specified topic number.

    Args:
        df (DataFrame): The pandas DataFrame containing the topics and representative documents, obtained by `topic_model.get_topic_info()`.
        topic_num (int): The topic number for which representative topics should be printed.

    Returns:
        None

    Example:
        >>> print_representative_topics(df, 2)
    """
    filtered_topic = df[df['Topic'] == topic_num].iloc[0]
    print(f"Representative docs for Topic {topic_num}, that is \"{filtered_topic['Name']}\":")
    for doc in filtered_topic['Representative_Docs']:
        print(doc)
        print("****************")


In [ ]:
# Example usage:
topic_num = 3
print_representative_topics(df_topics, topic_num)


In [ ]:
#| export
def get_topic(topic_model, topic_num):
    print('\n'.join(L(topic_model.get_topic(topic_num, full=False)).map(lambda pair: f"{pair[0]} --> {pair[1]}")))

In [ ]:
# Example usage
get_topic(topic_model, topic_num)

Look at a random document and figure if it belongs to the correct topic:

In [ ]:
df_docs = topic_model.get_document_info(tokenized_docs)
df_docs.sample(1)

## Search Topics:

Create some dummy product descriptions to search for most relevant topics:

In [ ]:
product_description = """
Introducing our latest collection of men's casual shirts, designed to elevate your everyday style with effortless sophistication. Crafted from premium quality cotton fabric, our shirt offers unrivaled comfort and breathability, making it perfect for all-day wear.

Featuring a classic button-down design with a modern twist, this shirt combines timeless elegance with contemporary flair. The slim-fit silhouette ensures a sleek and tailored look, while the versatile color options make it easy to pair with any outfit.

Whether you're heading to the office, meeting friends for brunch, or enjoying a night out on the town, this shirt will keep you looking and feeling your best. Dress it up with tailored trousers and loafers for a polished ensemble, or keep it casual with jeans and sneakers for a laid-back vibe.

Upgrade your wardrobe with our stylish men's shirt and experience the perfect blend of comfort, quality, and style. Add it to your cart today and elevate your wardrobe essentials with ease.
"""

album_description = """
Introducing our latest music album, an eclectic blend of captivating melodies and soul-stirring rhythms that will take you on a musical journey like no other. With a unique fusion of genres ranging from jazz and blues to funk and soul, this album promises to delight music lovers of all tastes.

Featuring mesmerizing vocals, skillful instrumentation, and heartfelt lyrics, each track on this album tells a story and evokes a range of emotions. From upbeat anthems that will have you dancing along to heartfelt ballads that will tug at your heartstrings, there's something for everyone on this album.

Whether you're looking to unwind after a long day, set the mood for a romantic evening, or simply groove to some infectious beats, this album has you covered. Let the music transport you to new and exciting places, and allow yourself to get lost in its enchanting melodies.

Experience the magic of our latest music album and discover why it's destined to become a timeless classic. Add it to your playlist today and prepare to be captivated from the very first note.
"""

glove_description = """
GripMaster Elite Baseball Gloves Experience unmatched comfort and control with the GripMaster Elite Baseball Gloves. 

Designed for the serious player, these gloves are made from high-quality leather, ensuring durability and a soft, supple feel. The tailored fit adapts to your hand shape, enhancing flexibility and grip. 

Strategic perforations increase breathability, keeping your hands cool under pressure. 

Whether you're catching fastballs or fielding grounders, the GripMaster Elite provides the support and tactile sensitivity you need to perform at your best.
"""

In [ ]:
#| export
def print_top_topics(topic_model, description, topics_map, top_n=5):
    """
    Prints the top N topics related to a given description.

    Args:
    - topic_model (BERTopic): A BERTopic model instance.
    - description (str): The description for which top topics are to be found.
    - topics_map (dict): A dictionary mapping topic numbers to their corresponding names.
    - top_n (int, optional): The number of top topics to print. Defaults to 5.

    Returns:
    None
    """
    similar_topics, similarity = topic_model.find_topics(description, top_n=top_n)
    print(f'The top {top_n} topics for the description are:')
    for i, (topic, sim) in enumerate(zip(similar_topics, similarity)):
        print(f"{i}: {topics_map.get(topic, None)}, similarity is {sim}")
        get_topic(topic_model, topic)
        print("**********************")


In [ ]:
# Example usage
print_top_topics(topic_model, glove_description, topics_map)

In [ ]:
topic_model.topic_aspects_

## Serialization

In [ ]:
#| export
def save_topic_model(topic_model, path, embedding_model_name=None):
    """
    Saves the topic model to the specified path, including the embedding model.

    Args:
    - topic_model (BERTopic): A BERTopic model instance.
    - path (str): The path where the model will be saved.
    - embedding_model: The embedding model used in the topic model.

    Returns:
    None
    """
    if embedding_model_name: print(f"Saving the embedding model {embedding_model_name} along with topic_model.")
    topic_model.save(path, serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model_name)

In [ ]:
#| eval: false 
import tempfile
with tempfile.TemporaryDirectory() as tmp_dir:
        # Define a dummy topic model and embedding model
        
        # Call the save_topic_model function with the temporary directory path
        save_topic_model(topic_model, tmp_dir, embedding_model)
        print('Saved')

        files = os.listdir(tmp_dir) 
        print(files)
        
        # Check if the model files are saved in the temporary directory
        for f in files:
                assert(os.path.exists(os.path.join(tmp_dir, f)))

In [ ]:
#| eval: false
#| hide
# Make a temp directory and save it so save it so that we can load later
model_dir = Path.cwd()/'tmp/lf_amazon_131k'
model_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
save_topic_model(topic_model, model_dir, embedding_model)

In [ ]:
# Load the saved topic_model with the embedding model that was used
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
loaded_model = BERTopic.load(model_dir, embedding_model=embedding_model)
topics_map = get_topicsmap(loaded_model.get_topic_info())
topic = 0
print(f"Loading saved topicmodel to check... Printing topic {topic}, that is \"{topics_map[topic]}\"")
get_topic(loaded_model, topic)

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()